In [ ]:
# Solar S1.0 repaired | Dariush 0.39 and optional exact MAGFiLO Blend V10 probabilities
# Mount the competition and dariushafshar/solar-filament-unet-weights V2 (ID 20868460).
# Recommended GPU: T4 x2. Replay works without a GPU. Fresh run keeps the author fallback in submission.csv.
# Optional: attach a blend_v10_manifest.json and its float32 .npy maps; a submission CSV cannot be used here.
# Change CONFIG if needed. Run this cell once. See SOLAR_S1_RUN.md for outputs and data contract.
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile

CONFIG = {'input_root': '/kaggle/input',
 'output_root': '/kaggle/working/Solar_S1_0',
 'kaggle_submission_path': '/kaggle/working/submission.csv',
 'selected_submission': 'S0',
 'scratch_root': '/kaggle/temp/Solar_S1_0',
 'dariush_receipt': '',
 'p039_input_cache': '',
 'blend_manifest': '',
 'audit_only': False,
 'replay_only': False,
 'install_missing_pycocotools': True,
 'max_gpus': 2,
 'seed': 2026,
 'oof_photos': 40,
 'alphas': [0.0, 0.15, 0.3, 0.45],
 'primary_alpha': 0.3,
 'feather_px': 64,
 'blend_keep_candidates': [[0.35, 0.9, 200, 0.72], [0.35, 0.84, 200, 0.68], [0.35, 0.94, 200, 0.76]],
 'min_calibration_dates': 8,
 'thresholds': [0.3, 0.35, 0.45, 0.55, 0.65],
 'candidate_min_area': 64,
 'instance_cost': 0.72,
 'repair_min_probability': 0.35,
 'repair_max_fraction': 0.1,
 'small_area': 500,
 'structural_iou_link': 0.1,
 'pixel_samples': 16384}
CODE_SHA256 = 'dcfac224e2a0dbb09bf8f4cc6f618663bc2d4edc5619057ace88a8db5d593632'

MODULES = {
    'diagnostics': """\"\"\"Official pooled PQ, candidate attribution, probability diagnostics and example panels.\"\"\"

from collections import defaultdict
import gzip
import json
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw
from pycocotools import mask as coco
from scipy import ndimage as ndi
from scipy.special import expit, logit
from scipy.optimize import minimize

from official_pq import canonical_rle, from_iou, pairwise_iou
from reconstruct import S8


def annotations(path, shape):
    raw = json.loads(path.read_text(encoding="utf-8-sig"))
    entries = {}
    for row in raw["images"]:
        stem = Path(row["file_name"]).stem
        if (int(row.get("height") or shape[0]), int(row.get("width") or shape[1])) != shape:
            raise ValueError(f"Unexpected annotation canvas for {stem}")
        record_id = str(row["id"])
        if record_id in entries:
            raise ValueError(f"Duplicate annotation record: {record_id}")
        entries[record_id] = (stem, row["file_name"], [])
    for row in raw["annotations"]:
        seg = row.get("segmentation")
        if not seg:
            continue
        stem, _, masks = entries[str(row["image_id"])]
        if isinstance(seg, dict):
            mask = canonical_rle(seg)
        else:
            mask = coco.merge(coco.frPyObjects(seg, *shape))
        if tuple(mask["size"]) != shape:
            raise ValueError(f"Bad RLE canvas for {stem}")
        if int(coco.area(mask)) > 0:
            masks.append(mask)
    records, names = defaultdict(dict), {}
    for record_id, (stem, filename, masks) in entries.items():
        if names.setdefault(stem, filename) != filename:
            raise ValueError(f"Conflicting photo paths for {stem}")
        records[stem][record_id] = masks
    return records, names


def evaluate_photo(records, selected, pool=None, small_area=500, structural_link=.1):
    totals = defaultdict(float)
    for gt in records.values():
        iou = pairwise_iou(gt, selected)
        result = from_iou(iou)
        for key in ("tp", "fp", "fn", "iou_sum", "ambiguous_pred", "ambiguous_gt"):
            totals[key] += getattr(result, key)
        link = iou >= structural_link
        totals["fragmented_gt"] += int((link.sum(axis=1) > 1).sum())
        totals["merged_pred"] += int((link.sum(axis=0) > 1).sum())
        areas = np.array([int(coco.area(mask)) for mask in gt])
        totals["small_fn"] += int((((iou > .5).sum(axis=1) == 0) & (areas < small_area)).sum())
        totals["gt_views"] += len(gt)
        if pool is not None:
            oracle = pairwise_iou(gt, pool).max(axis=1, initial=0)
            totals["oracle_hits"] += int((oracle > .5).sum())
            totals["oracle_best_iou_sum"] += float(oracle.sum())
    return scored(totals)


SUM_FIELDS = ("tp", "fp", "fn", "iou_sum", "ambiguous_pred", "ambiguous_gt", "fragmented_gt", "merged_pred",
              "small_fn", "gt_views", "oracle_hits", "oracle_best_iou_sum")


def scored(totals):
    totals = {k: totals.get(k, 0) for k in SUM_FIELDS}
    tp, fp, fn = (totals[k] for k in ("tp", "fp", "fn"))
    denom = tp + .5 * (fp + fn)
    gt = totals["gt_views"]
    return totals | {"pq": totals["iou_sum"] / denom if denom else 0.,
                     "sq": totals["iou_sum"] / tp if tp else 0., "rq": tp / denom if denom else 0.,
                     "oracle_recall": totals["oracle_hits"] / gt if gt else None}


def pooled(rows):
    return scored({key: sum(row[key] for row in rows) for key in SUM_FIELDS})


def sample_center(records, shape, crop, count, rng):
    y0, y1, x0, x1 = crop
    yy = rng.integers(y0, y1, size=count)
    xx = rng.integers(x0, x1, size=count)
    target = np.zeros(count, np.float32)
    for gt in records.values():
        if gt:
            target += coco.decode(coco.merge(gt))[yy, xx]
    return yy, xx, target / len(records)


def calibration_fit(prob, truth):
    \"\"\"Fit a bounded logit scale and offset against independent annotator fractions.\"\"\"
    z = logit(np.clip(prob.astype(np.float64), 1e-5, 1 - 1e-5))

    def loss(params):
        a, b = params
        pred = a * z + b
        return float(np.mean(np.logaddexp(0, pred) - truth * pred) + .01 * ((a - 1)**2 + b**2))

    result = minimize(loss, [1., 0.], bounds=[(.75, 1.35), (-.6, .6)], method="L-BFGS-B")
    return tuple(map(float, result.x)) if result.success else (1., 0.)


def apply_calibration(pb, params):
    if params == (1., 0.):
        return pb
    return expit(params[0] * logit(np.clip(pb, 1e-5, 1 - 1e-5)) + params[1]).astype(np.float32)


def reliability(prob, truth=None, bins=20):
    ids = np.minimum((prob * bins).astype(np.int32), bins - 1).ravel()
    count = np.bincount(ids, minlength=bins)
    mass = np.bincount(ids, weights=prob.ravel(), minlength=bins)
    out = {"n": int(prob.size), "histogram": count.tolist(), "bin_probability_sum": mass.tolist()}
    if truth is not None:
        expected = np.bincount(ids, weights=truth.ravel(), minlength=bins)
        out.update(brier=float(np.mean((prob - truth)**2)), ece=float(np.abs(expected - mass).sum() / prob.size),
                   bin_truth_sum=expected.tolist())
    return out


def complement(p039, pb, fused, crop, low):
    y0, y1, x0, x1 = crop
    center = p039[y0:y1, x0:x1]
    a, b, c = center > low, pb > low, fused[y0:y1, x0:x1] > low
    first, before = ndi.label(p039 > low, structure=S8)
    second, after = ndi.label(fused > low, structure=S8)
    both = (first > 0) & (second > 0)
    pair, overlap = np.unique(first[both].astype(np.int64) * (after + 1) + second[both], return_counts=True)
    left, right = np.divmod(pair, after + 1)
    area_first = np.bincount(first.ravel())
    area_second = np.bincount(second.ravel())
    meaningful = overlap >= .10 * np.minimum(area_first[left], area_second[right])
    links_first = np.bincount(left[meaningful], minlength=before+1)[1:]
    links_second = np.bincount(right[meaningful], minlength=after+1)[1:]
    return {"threshold": low, "p039_only_pixels": int((a & ~b).sum()), "blend_only_pixels": int((b & ~a).sum()),
            "agree_fg_pixels": int((a & b).sum()), "mean_absolute_gap": float(np.abs(center - pb).mean()),
            "fused_added_pixels": int((c & ~a).sum()), "fused_lost_pixels": int((a & ~c).sum()),
            "cc_before": before, "cc_after": after, "cc_split": int((links_first > 1).sum()),
            "cc_merged": int((links_second > 1).sum()), "cc_disappeared": int((links_first == 0).sum()),
            "cc_appeared": int((links_second == 0).sum())}


def mask_counts(rle):
    return rle["counts"].decode("ascii")


def candidate_record(node, selected):
    return {"id": node.key, "level": node.level, "threshold": node.threshold,
            "bbox_yxyx": [v for axis in node.box for v in (axis.start, axis.stop)], "area": node.area,
            "mean": node.mean, "peak": node.peak, "q90": node.q90, "high_fraction": node.high_fraction,
            "pca_length": node.length, "pca_width": node.width, "elongation": node.elongation,
            "boundary_contrast": node.contrast, "stability": node.stability, "agreement": node.agreement,
            "agreement_coverage": node.agreement_coverage, "basic_utility": node.basic,
            "quality_utility": node.quality, "parent_index": node.parent, "children_indices": node.children,
            "selected_in": selected, "segmentation_rle": mask_counts(node.rle)}


def save_candidates(path, nodes, stages):
    memberships = defaultdict(list)
    changes, previous = {}, set()
    for stage, rows in stages.items():
        selected = {node.key for node, _, _ in rows}
        for node_id in selected:
            memberships[node_id].append(stage)
        changes[stage] = {"selected_ids": sorted(selected), "added_ids": sorted(selected - previous),
                          "removed_ids": sorted(previous - selected),
                          "repaired": [{"id": node.key, "added_pixels": added, "segmentation_rle": mask_counts(rle)}
                                       for node, rle, added in rows if added]}
        # Both closing arms compare directly with the same unmodified quality selection.
        if stage not in {"small_close", "selective_close", "quality_no_agreement"}:
            previous = selected
    with gzip.open(path, "wt", encoding="utf-8", compresslevel=3) as stream:
        json.dump({"nodes": [candidate_record(n, memberships[n.key]) for n in nodes], "stage_changes": changes},
                  stream, allow_nan=False)
    return changes


def area_stats(masks):
    values = [int(coco.area(m)) for m in masks]
    return {"instances": len(values), "pixels": sum(values), "areas": values,
            "median_area": float(np.median(values)) if values else None}


def require_disjoint_fast(masks, shape):
    if any(tuple(mask["size"]) != shape for mask in masks):
        raise ValueError("Submission canvas mismatch")
    if not masks:
        return
    areas = [int(coco.area(mask)) for mask in masks]
    if min(areas) == 0 or sum(areas) != int(coco.area(coco.merge(masks))):
        raise ValueError("Submission contains empty or overlapping masks")


def render_panel(path, stem, gray, prob_panels, mask_panels):
    size = 384
    gray_img = Image.fromarray(gray).resize((size, size), Image.Resampling.BILINEAR)
    gray_small = np.asarray(gray_img)
    panels = [("Observation", gray_img.convert("RGB"))]
    for title, prob in prob_panels:
        encoded = Image.fromarray(np.rint(255 * prob).astype(np.uint8)).resize((size, size), Image.Resampling.BILINEAR)
        heat = np.asarray(encoded, dtype=np.float32) / 255
        rgb = np.stack([gray_small * (1 - .65 * heat) + 165 * heat,
                        gray_small * (1 - .75 * heat) + 65 * heat, gray_small * (1 - .9 * heat)], axis=-1)
        panels.append((title, Image.fromarray(np.clip(rgb, 0, 255).astype(np.uint8))))
    colors = ((79, 237, 172), (255, 192, 71), (99, 185, 254), (235, 132, 230))
    for title, masks in mask_panels:
        rgb = np.asarray(gray_img.convert("RGB")).copy()
        for i, mask in enumerate(masks):
            dense = coco.decode(mask)
            scaled = np.asarray(Image.fromarray(dense * 255).resize((size, size), Image.Resampling.BOX)) > 32
            edge = scaled & ~ndi.binary_erosion(scaled)
            rgb[edge] = colors[i % len(colors)]
        panels.append((title, Image.fromarray(rgb)))
    cols = min(3, len(panels))
    canvas = Image.new("RGB", (cols * size, ((len(panels)+cols-1)//cols) * (size + 26)), (21, 24, 30))
    draw = ImageDraw.Draw(canvas)
    for i, (title, panel) in enumerate(panels):
        x, y = (i % cols) * size, (i // cols) * (size + 26)
        draw.text((x + 8, y + 7), f"{stem}  {title}", fill=(240, 245, 250))
        canvas.paste(panel, (x, y + 26))
    canvas.save(path)
""",
    'official_pq': """\"\"\"Official-notebook-compatible pooled PQ. No model or dataset downloads.

Reference: azimahmadzadeh/self-evaluation-notebook, version 6, retrieved
2026-11-02. The reference counts ALL IoU > 0.5 pairs, not Hungarian matches.
This module deliberately preserves that behavior, including overlapping GT.
\"\"\"
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Mapping, Sequence

import numpy as np
from pycocotools import mask as coco_mask


@dataclass
class Totals:
    tp: int = 0
    fp: int = 0
    fn: int = 0
    iou_sum: float = 0.0
    ambiguous_gt: int = 0
    ambiguous_pred: int = 0
    fragmented_gt: int = 0
    merged_pred: int = 0

    def __add__(self, other: Totals) -> Totals:
        return Totals(**{k: v + asdict(other)[k] for k, v in asdict(self).items()})

    def summary(self) -> dict:
        denominator = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return {
            **asdict(self),
            "pq": self.iou_sum / denominator if denominator else 0.0,
            "dq": self.tp / denominator if denominator else 0.0,
            "sq": self.iou_sum / self.tp if self.tp else 0.0,
        }


def from_iou(ious: np.ndarray) -> Totals:
    \"\"\"Count exactly as the host notebook; diagnostics use any positive IoU.\"\"\"
    ious = np.asarray(ious, dtype=np.float64)
    if ious.ndim != 2 or not np.isfinite(ious).all():
        raise ValueError("IoU matrix must be a finite 2-D array")
    if ((ious < 0) | (ious > 1)).any():
        raise ValueError("IoU values must be in [0, 1]")
    hit = ious > 0.5
    links = ious > 0
    return Totals(
        tp=int(hit.sum()),
        fp=int((hit.sum(axis=0) == 0).sum()),
        fn=int((hit.sum(axis=1) == 0).sum()),
        iou_sum=float(ious[hit].sum()),
        ambiguous_gt=int((hit.sum(axis=1) > 1).sum()),
        ambiguous_pred=int((hit.sum(axis=0) > 1).sum()),
        fragmented_gt=int((links.sum(axis=1) > 1).sum()),
        merged_pred=int((links.sum(axis=0) > 1).sum()),
    )


def canonical_rle(rle: dict) -> dict:
    size = list(map(int, rle["size"]))
    if len(size) != 2 or min(size) <= 0:
        raise ValueError("RLE size must be positive height,width")
    counts = rle["counts"]
    if isinstance(counts, str):
        counts = counts.encode("ascii")
    if isinstance(counts, list):
        return coco_mask.frPyObjects({"size": size, "counts": counts}, *size)
    return {"size": size, "counts": counts}


def pairwise_iou(gt: Sequence[dict], pred: Sequence[dict]) -> np.ndarray:
    if not gt or not pred:
        return np.zeros((len(gt), len(pred)), dtype=np.float64)
    gt, pred = list(map(canonical_rle, gt)), list(map(canonical_rle, pred))
    shapes = {tuple(r["size"]) for r in gt + pred}
    if len(shapes) != 1:
        raise ValueError("GT and prediction RLE sizes differ; resize before encoding")
    return np.asarray(coco_mask.iou(pred, gt, [0] * len(gt)), dtype=np.float64).T


def require_disjoint(pred: Sequence[dict], shape=(2048, 2048)) -> None:
    \"\"\"Reject invalid submissions rather than silently changing their masks.\"\"\"
    occupied = np.zeros(shape, dtype=bool)
    for r in pred:
        r = canonical_rle(r)
        if tuple(r["size"]) != tuple(shape):
            raise ValueError("Prediction has unexpected canvas size")
        current = coco_mask.decode(r).astype(bool)
        if not current.any():
            raise ValueError("Empty mask is not a filament instance")
        if (occupied & current).any():
            raise ValueError("Prediction instances overlap")
        occupied |= current


def evaluate_records(
    records: Mapping[str, tuple[str, Sequence[dict]]],
    predictions: Mapping[str, Sequence[dict]],
    *, validate_predictions: bool = True, shape=(2048, 2048),
) -> dict:
    \"\"\"One prediction set per photo, scored against EVERY annotator separately.

    records: {annotator_record_id: (physical_photo_stem, list_of_GT_RLEs)}.
    Missing predictions are [], and unknown prediction photos are rejected.
    Explicit zero-GT records are supported using the host score's empty-row
    rule. Unlike its CSV-only entry discovery, the manifest retains such rows.
    \"\"\"
    known = {stem for stem, _ in records.values()}
    unknown = set(predictions) - known
    if unknown:
        raise ValueError(f"Predictions outside evaluation split: {sorted(unknown)[:3]}")
    for stem in known:
        if validate_predictions:
            require_disjoint(predictions.get(stem, []), shape)
    total = Totals()
    by_photo: dict[str, Totals] = {}
    by_record = {}
    nonzero_ious, matched_ious = [], []
    for record_id, (stem, gt) in records.items():
        pred = predictions.get(stem, [])
        for r in list(gt) + list(pred):
            if tuple(r["size"]) != tuple(shape):
                raise ValueError("RLE canvas differs from evaluation canvas")
        ious = pairwise_iou(gt, pred)
        result = from_iou(ious)
        total = total + result
        by_photo[stem] = by_photo.get(stem, Totals()) + result
        by_record[record_id] = result.summary()
        nonzero_ious.extend(ious[ious > 0].tolist())
        matched_ious.extend(ious[ious > 0.5].tolist())
    return {
        "pooled": total.summary(),
        "n_photos": len(known), "n_annotator_records": len(records),
        "n_prediction_instances_physical": sum(len(predictions.get(s, [])) for s in known),
        "per_photo": {s: t.summary() for s, t in sorted(by_photo.items())},
        "per_record": by_record,
        "positive_pair_ious": nonzero_ious,
        "positive_pair_dices": [2 * x / (1 + x) for x in nonzero_ious],
        "matched_pair_ious": matched_ious,
        "notes": "Pooled all-pairs host semantics; no one-to-one assignment. Empty total returns 0.",
    }
""",
    'probability': """\"\"\"Dariush 8TTA inference and validated probability cache inputs.\"\"\"

from concurrent.futures import ThreadPoolExecutor
import csv
import hashlib
import importlib.metadata
import json
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
from threading import Lock
from zipfile import BadZipFile

import numpy as np
from PIL import Image

from reconstruct import check_probability


SHAPE = (2048, 2048)
CROP = (88, 1960, 88, 1960)
D4 = [(k, flip) for k in (0, 2, 1, 3) for flip in (False, True)]
ENCODERS = {"resnet34", "tu-tf_efficientnetv2_s"}
LEGACY_RECIPE = "dariush_8d4_native_disk950_amp16_permodel_u8_and_float32_v1"
RECIPE = "dariush_8d4_native_disk950_amp16_permodel_u8_fp32_pixel_repair_v2"
BAD_CACHE = (ValueError, TypeError, KeyError, OSError, EOFError, BadZipFile)
BLEND_VERSION = 344704584
BLEND_KERNEL = "anthonytherrien/magfilo-blended-tile-u-net-for-solar-filaments"


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def write_json(path, value):
    path = Path(path)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as f:
        temp = Path(f.name)
        json.dump(value, f, indent=2, allow_nan=False)
        f.write("\\n")
    temp.replace(path)


def save_npz(path, **arrays):
    path = Path(path)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as stream:
        temp = Path(stream.name)
        np.savez_compressed(stream, **arrays)
    temp.replace(path)


def discover_039(root, receipt_setting):
    paths = [Path(receipt_setting)] if receipt_setting else sorted(root.rglob("solar_unet_receipt.json"))
    if len(paths) != 1 or not paths[0].is_file():
        raise FileNotFoundError(f"Mount Dariush solar-filament-unet-weights V2 (receipt found at {paths})")
    path = paths[0]
    receipt = read_json(path)
    models = receipt["ensemble"]["models"]
    if len(models) != 10 or {(m["encoder"], int(m["fold"])) for m in models} != {
            (encoder, fold) for encoder in ENCODERS for fold in range(5)}:
        raise ValueError("The 0.39 receipt must contain both encoders for each of the five folds")
    for model in models:
        model["fold"] = int(model["fold"])
        expected = f"unet_{model['encoder']}_fold{model['fold']}.pt"
        if model["file"] != expected or len(model["sha256"]) != 64:
            raise ValueError(f"Unexpected checkpoint in the receipt: {model}")
    fold_file = path.parent / "folds.csv"
    with fold_file.open(newline="", encoding="utf-8-sig") as stream:
        rows = list(csv.DictReader(stream))
    folds = {row["stem"]: int(row["fold"]) for row in rows}
    if len(rows) != 707 or len(folds) != 707 or set(folds.values()) != set(range(5)):
        raise ValueError("Expected the author's 707 physical-photo fold assignments")
    return path, receipt, models, folds


def gray_image(path):
    with Image.open(path) as image:
        gray = np.asarray(image.convert("L"), dtype=np.uint8)
    if gray.shape != SHAPE:
        raise ValueError(f"Unexpected photo size: {path}, {gray.shape}")
    return gray


def metadata_for(stem, image_sha, models, fold, receipt_sha, folds_sha):
    chosen = [m["sha256"] for m in models if fold is None or m["fold"] == fold]
    return {"stem": stem, "image_sha256": image_sha, "recipe": RECIPE, "shape": list(SHAPE),
            "fold": fold, "receipt_sha256": receipt_sha, "folds_sha256": folds_sha, "models": chosen}


def validate_cache(pack, expected, path):
    actual = json.loads(pack["provenance"].item())
    # Old full caches passed the same finite/range check. Their untouched maps are reusable.
    if actual != expected and actual != (expected | {"recipe": LEGACY_RECIPE}):
        raise ValueError(f"Cache provenance mismatch: {path}")


def validate_original(original, path):
    if original.shape != SHAPE or original.dtype != np.uint8:
        raise ValueError(f"Invalid original 0.39 probability bytes in {path}")
    return original


def load_p039(path, expected):
    with np.load(path, allow_pickle=False) as pack:
        validate_cache(pack, expected, path)
        return check_probability(pack["p039"], SHAPE), validate_original(pack["baseline_u8"], path)


def load_s0(path, expected):
    with np.load(path, allow_pickle=False) as pack:
        validate_cache(pack, expected, path)
        return validate_original(pack["baseline_u8"], path)


def cache_record(path, has_float):
    with np.load(path, allow_pickle=False) as pack:
        events = json.loads(pack["precision_events"].item()) if "precision_events" in pack else []
        return {"path": str(path), **json.loads(pack["provenance"].item()),
                "has_float": has_float, "precision_events": events}


def resolve_cached(stems, cache, mounted, meta):
    floats, native = {}, {}
    for stem in stems:
        roots = [cache] if mounted is None else [cache, mounted]
        for base in roots:
            path = base / f"{stem}.npz"
            if path.is_file():
                try:
                    load_p039(path, meta[stem])
                except BAD_CACHE as exc:
                    print(f"Ignoring unusable float cache {path}: {exc}", flush=True)
                else:
                    floats[stem] = native[stem] = path
                    break
            if stem in native:
                continue
            for s0_path in (base / f"{stem}.s0.npz", path):
                if s0_path.is_file():
                    try:
                        load_s0(s0_path, meta[stem])
                    except BAD_CACHE as exc:
                        print(f"Ignoring unusable baseline cache {s0_path}: {exc}", flush=True)
                    else:
                        native[stem] = s0_path
                        break
    return floats, native


def ensure_p039(cache, photos, meta, receipt_path, models, settings):
    cache.mkdir(parents=True, exist_ok=True)
    mounted = Path(settings["p039_input_cache"]) if settings["p039_input_cache"] else None
    floats, native = resolve_cached(photos, cache, mounted, meta)
    pending = [stem for stem in photos if stem not in floats]
    replayed = len(floats)
    print(f"P039: {replayed} cached, {len(pending)} missing ({len(native) - replayed} with S0 only)", flush=True)
    runtime = {}
    if pending and not settings["replay_only"]:
        runtime, new_floats, new_native = infer_pending(cache, pending, photos, meta, receipt_path, models, settings)
        floats.update(new_floats)
        native.update(new_native)
    runtime.update(replayed=replayed, computed=len(floats) - replayed,
                   missing_float=[s for s in photos if s not in floats],
                   missing_s0=[s for s in photos if s not in native])
    return floats, native, runtime


def invalid_pixels(prob):
    return ~np.isfinite(prob) | (prob < 0) | (prob > 1)


def repair_probability(raw, retry_fp32, oom_error):
    \"\"\"Repair only invalid FP16 pixels; preserve the existing uint8 S0 path.\"\"\"
    invalid = invalid_pixels(raw)
    count = int(invalid.sum())
    if not count:
        return raw, None
    event = {"initial_invalid": count, "nonfinite": int((~np.isfinite(raw)).sum()),
             "repaired": 0, "remaining": count, "retry": "fp32_8tta_single_view"}
    try:
        alternative = retry_fp32()
    except oom_error:
        event["retry"] = "fp32_out_of_memory"
        return raw, event
    except RuntimeError as exc:
        event.update(retry="fp32_runtime_error", error=str(exc))
        return raw, event
    valid = invalid & ~invalid_pixels(alternative)
    np.copyto(raw, alternative, where=valid)
    event["repaired"] = int(valid.sum())
    event["remaining"] -= event["repaired"]
    return raw, event


def save_completed_photo(cache, stem, meta, float_sum, integer_sum, count, events):
    original = np.rint(integer_sum.astype(np.float32) / count).astype(np.uint8)
    raw = None
    if not any(event["remaining"] for event in events):
        raw = (float_sum / count).astype(np.float32)
        try:
            check_probability(raw, SHAPE)
        except ValueError:
            events = [*events, {"retry": "ensemble_validation_failed", "remaining": int(invalid_pixels(raw).sum())}]
            raw = None
    payload = {"baseline_u8": original, "provenance": np.array(json.dumps(meta, sort_keys=True)),
               "precision_events": np.array(json.dumps(events, sort_keys=True))}
    if raw is None:
        path = cache / f"{stem}.s0.npz"
        save_npz(path, **payload)
        print(f"{stem}: float map unavailable; saved original S0 bytes in {path.name}", flush=True)
        return None, path
    path = cache / f"{stem}.npz"
    save_npz(path, p039=raw, **payload)
    return path, path


def predict_tta(torch, model, photo, device, chunk, amp=True):
    pixels, mean, std = photo
    with torch.inference_mode():
        x = torch.from_numpy(np.array(pixels, dtype=np.uint8, copy=True)).to(device).float().div_(255.)
        z = ((x - mean) / std)[None, None]
        acc = torch.zeros(SHAPE, dtype=torch.float32, device=device)
        for j in range(0, len(D4), chunk):
            ops = D4[j:j + chunk]
            inp = torch.cat([torch.rot90(z.flip((3,)) if flip else z, k, (2, 3)) for k, flip in ops])
            inp = inp.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model(inp)
            for p, (k, flip) in zip(logits.float().sigmoid(), ops, strict=True):
                p = torch.rot90(p[0], -k, (0, 1))
                acc += p.flip((1,)) if flip else p
        acc /= len(D4)
        if not amp:
            return acc.cpu().numpy()
        return acc.cpu().numpy(), (acc * 255.).round_().clamp_(0, 255).to(torch.uint8).cpu().numpy()


def infer_pending(cache, pending, photos, meta, receipt_path, models, settings):
    import importlib
    import time
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU (T4 x2 recommended) or mount the complete P039 cache")
    base = receipt_path.parent
    for model in models:
        if sha256(base / model["file"]) != model["sha256"]:
            raise ValueError(f"Dariush checkpoint SHA-256 differs from receipt: {model['file']}")
    try:
        version = importlib.metadata.version("segmentation-models-pytorch")
    except importlib.metadata.PackageNotFoundError:
        version = None
    if version != "0.5.0":
        wheel = base / "segmentation_models_pytorch-0.5.0-py3-none-any.whl"
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheel)], check=True)
        importlib.invalidate_caches()
    import segmentation_models_pytorch as smp
    import timm

    seed = settings["seed"]
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True  # Match the author. Record environment; byte parity is not promised.
    yy, xx = np.ogrid[:SHAPE[0], :SHAPE[1]]
    disk = (yy - (SHAPE[0] - 1) / 2)**2 + (xx - (SHAPE[1] - 1) / 2)**2 <= 950**2

    def prepare(stem):
        pixels = gray_image(photos[stem])
        region = pixels[disk].astype(np.float32) / 255.
        return pixels, float(region.mean()), float(np.float32(max(float(region.std()), 1e-3)))

    with ThreadPoolExecutor(8) as pool:
        inputs = list(pool.map(prepare, pending))
    scratch = Path(settings["scratch_root"])
    scratch.mkdir(parents=True, exist_ok=True)
    bytes_needed = len(pending) * int(np.prod(SHAPE)) * (8 + 2)
    if shutil.disk_usage(scratch).free < bytes_needed + 256 * 1024**2:
        raise OSError(f"Temporary inference needs approximately {bytes_needed / 1e9:.1f} GB free in {scratch}")
    devices = [torch.device(f"cuda:{i}") for i in range(min(settings["max_gpus"], torch.cuda.device_count()))]
    if not devices:
        raise ValueError("max_gpus must be positive")

    def predict_retry(model, photo, device, chunk):
        while True:
            try:
                return predict_tta(torch, model, photo, device, chunk), chunk
            except torch.cuda.OutOfMemoryError:
                if chunk == 1:
                    raise
            chunk //= 2
            torch.cuda.empty_cache()
            print(f"{device}: retry with {chunk} views per pass, still eight views total", flush=True)

    lock = Lock()
    todo = iter(models)
    counts = np.zeros(len(pending), np.uint8)
    expected = [len(meta[s]["models"]) for s in pending]
    logs, new_floats, new_native = {}, {}, {}
    events = {stem: [] for stem in pending}
    with tempfile.TemporaryDirectory(prefix="p039_", dir=scratch) as tmp:
        floats = np.memmap(Path(tmp) / "float.dat", mode="w+", dtype=np.float64, shape=(len(pending), *SHAPE))
        integers = np.memmap(Path(tmp) / "integer.dat", mode="w+", dtype=np.uint16, shape=(len(pending), *SHAPE))
        floats[:] = 0
        integers[:] = 0

        def worker(device):
            while True:
                with lock:
                    row = next(todo, None)
                if row is None:
                    return
                start = time.monotonic()
                chosen = [i for i, s in enumerate(pending) if meta[s]["fold"] in (None, row["fold"])]
                if not chosen:
                    continue
                model = smp.Unet(encoder_name=row["encoder"], encoder_weights=None, in_channels=1, classes=1)
                model.load_state_dict(torch.load(base / row["file"], map_location="cpu", weights_only=True), strict=True)
                model = model.to(device).to(memory_format=torch.channels_last).eval()
                chunk = 4
                for i in chosen:
                    stem = pending[i]
                    (raw, u8), chunk = predict_retry(model, inputs[i], device, chunk)
                    raw, event = repair_probability(raw, lambda: predict_tta(torch, model, inputs[i], device, 1, amp=False),
                                                    torch.cuda.OutOfMemoryError)
                    if event is not None:
                        event.update(stem=stem, checkpoint=row["file"], device=str(device), original_amp_batch=chunk)
                        print(f"P039 precision recovery: {event}", flush=True)
                        if event["retry"] == "fp32_out_of_memory":
                            torch.cuda.empty_cache()
                    with lock:
                        if event is not None:
                            events[stem].append(event)
                        if not any(item["remaining"] for item in events[stem]):
                            floats[i] += raw
                        integers[i] += u8
                        counts[i] += 1
                        complete = counts[i] == expected[i]
                    if complete:
                        p039_path, s0_path = save_completed_photo(cache, stem, meta[stem], floats[i], integers[i],
                                                                expected[i], events[stem])
                        with lock:
                            new_native[stem] = s0_path
                            if p039_path is not None:
                                new_floats[stem] = p039_path
                del model
                torch.cuda.empty_cache()
                with lock:
                    logs[row["file"]] = {"photos": len(chosen), "chunk": chunk, "device": str(device),
                                         "seconds": round(time.monotonic() - start, 1)}
                print(f"P039 {row['file']}: {logs[row['file']]}", flush=True)

        with ThreadPoolExecutor(len(devices)) as pool:
            jobs = [pool.submit(worker, device) for device in devices]
            for job in jobs:
                job.result()
        if list(counts) != expected or set(new_native) != set(pending):
            raise RuntimeError("The P039 inference coverage does not match its receipt")
        del floats, integers
    runtime = {"recipe": RECIPE, "smp": smp.__version__, "timm": timm.__version__, "torch": torch.__version__,
               "cuda": torch.version.cuda, "gpu": [torch.cuda.get_device_name(d) for d in devices], "models": logs,
               "precision_events": [event for stem in pending for event in sorted(events[stem], key=lambda e: e["checkpoint"])]}
    return runtime, new_floats, new_native


def blend_contract(path):
    data = read_json(path)
    expected = {"kernel": BLEND_KERNEL, "script_version_id": BLEND_VERSION, "canvas": list(SHAPE),
                "crop_yxyx": list(CROP), "coordinate_order": "yx", "probability_kind": "pre_threshold_foreground",
                "dtype": "float32", "tta_views": 8, "tile_shape": [1056, 1056], "tile_weighting": "cosine"}
    if any(data.get(key) != val for key, val in expected.items()):
        raise ValueError("Blend manifest does not describe native-coordinate V10 pre-threshold 8TTA probabilities")
    if not data.get("source_url") or not data.get("provenance_note") or not isinstance(data.get("images"), dict):
        raise ValueError("Blend needs a source URL, provenance note, and image roster")
    return data


def load_blend(base, item, image_sha):
    if item["image_sha256"] != image_sha:
        raise ValueError("Blend photo bytes differ from the official photo")
    name = item["file"]
    if Path(name).name != name or not name.endswith(".npy"):
        raise ValueError("Blend manifest requires a sibling .npy per photo")
    path = base / name
    if sha256(path) != item["probability_sha256"]:
        raise ValueError(f"Blend probability SHA-256 differs: {path}")
    pb = check_probability(np.load(path, allow_pickle=False), (CROP[1]-CROP[0], CROP[3]-CROP[2]))
    if not np.any((pb > 0) & (pb < 1)):
        raise ValueError("Blend input is binary or saturated, not a usable probability map")
    return pb
""",
    'reconstruct': """\"\"\"Semantic fusion and disjoint instance reconstruction for Solar S1.0.\"\"\"

from dataclasses import dataclass, field

import numpy as np
from pycocotools import mask as coco
from scipy import ndimage as ndi


S8 = np.ones((3, 3), bool)
KEEP_039 = (.35, .99, 200, .85)


def check_probability(array, shape):
    if array.dtype != np.float32 or array.shape != shape:
        raise ValueError(f"Expected float32 probability {shape}, got {array.dtype} {array.shape}")
    if not np.isfinite(array).all() or array.min() < 0 or array.max() > 1:
        raise ValueError("Probability contains nonfinite or out-of-range values")
    return array


def feather_weights(shape, crop, width):
    y0, y1, x0, x1 = crop
    if not (0 <= y0 < y1 <= shape[0] and 0 <= x0 < x1 <= shape[1] and width > 0):
        raise ValueError("Invalid Blend crop or feather width")
    y, x = np.ogrid[:y1 - y0, :x1 - x0]
    distance = np.minimum(np.minimum(y, y1 - y0 - 1 - y), np.minimum(x, x1 - x0 - 1 - x))
    return (.5 - .5 * np.cos(np.pi * np.minimum(distance / width, 1))).astype(np.float32)


def fuse(p039, pb, crop, feather, alpha):
    y0, y1, x0, x1 = crop
    if not 0 <= alpha <= 1 or feather.shape != pb.shape:
        raise ValueError("Invalid blend weight or feather shape")
    out = p039.copy()
    center = out[y0:y1, x0:x1]
    weight = np.float32(alpha) * feather
    valid = weight > 0
    center[valid] = (1 - weight[valid]) * center[valid] + weight[valid] * pb[valid]
    return out


def labels_at(prob, threshold):
    labels, count = ndi.label(prob > threshold, structure=S8)
    foreground = labels > 0
    ids, values = labels[foreground], prob[foreground]
    areas = np.bincount(ids, minlength=count + 1)
    sums = np.bincount(ids, weights=values, minlength=count + 1)
    peaks = np.zeros(count + 1, dtype=prob.dtype)
    np.maximum.at(peaks, ids, values)
    return labels, areas, sums / np.maximum(areas, 1), peaks


def encode_roi(mask, box, shape):
    \"\"\"Encode a bounded region directly in native COCO column-major coordinates.\"\"\"
    height, width = shape
    y, x = box
    edges = np.diff(np.pad(mask, ((1, 1), (0, 0))).T, axis=1)
    cols, rows = np.nonzero(edges)
    runs = ((cols + x.start) * height + rows + y.start).reshape(-1, 2)
    if not len(runs):
        raise ValueError("Cannot encode an empty candidate")
    join = runs[:-1, 1] == runs[1:, 0]
    starts = runs[np.r_[True, ~join], 0]
    ends = runs[np.r_[~join, True], 1]
    boundaries = np.r_[0, np.column_stack((starts, ends)).ravel()]
    if ends[-1] != height * width:
        boundaries = np.r_[boundaries, height * width]
    counts = np.diff(boundaries).tolist()
    return coco.frPyObjects({"size": list(shape), "counts": counts}, height, width)


def base_cc(prob, keep, quantized=False):
    low, high, min_area, min_mean = keep
    scale = 255 if quantized else 1
    low, high = (round(low * 255), round(high * 255)) if quantized else (low, high)
    labels, areas, means, peaks = labels_at(prob, low)
    boxes = ndi.find_objects(labels)
    ids = np.flatnonzero((areas >= min_area) & (means / scale >= min_mean) & (peaks > high))
    return [encode_roi(labels[boxes[i - 1]] == i, boxes[i - 1], prob.shape) for i in ids]


@dataclass
class Candidate:
    key: str
    level: int
    threshold: float
    box: tuple
    mask: np.ndarray
    area: int
    mean: float
    peak: float
    q90: float
    high_fraction: float
    length: float
    width: float
    elongation: float
    contrast: float
    agreement: float | None
    agreement_coverage: float
    rle: dict
    parent: int | None
    children: list = field(default_factory=list)
    stability: float = 0.
    basic: float = 0.
    quality: float = 0.


def expand(box, shape, padding):
    return tuple(slice(max(0, axis.start - padding), min(size, axis.stop + padding))
                 for axis, size in zip(box, shape))


def relative(box, outer):
    return tuple(slice(axis.start - base.start, axis.stop - base.start) for axis, base in zip(box, outer))


def candidates(prob, settings, p039=None, pb=None, crop=None):
    all_nodes, previous_labels, previous_nodes = [], None, {}
    thresholds = settings["thresholds"]
    if not thresholds or sorted(set(thresholds)) != list(thresholds):
        raise ValueError("Candidate thresholds must be strictly increasing")
    for level, threshold in enumerate(thresholds):
        labels, areas, means, peaks = labels_at(prob, threshold)
        boxes = ndi.find_objects(labels)
        current = {}
        for label in np.flatnonzero(areas >= settings["candidate_min_area"]):
            box = boxes[label - 1]
            mask = labels[box] == label
            area = int(areas[label])
            ys, xs = np.nonzero(mask)
            abs_y, abs_x = ys + box[0].start, xs + box[1].start
            parent = None if previous_labels is None else previous_nodes[int(previous_labels[abs_y[0], abs_x[0]])]
            var_y, var_x = np.var(ys), np.var(xs)
            covariance = np.mean((ys - ys.mean()) * (xs - xs.mean()))
            span = np.hypot(var_y - var_x, 2 * covariance)
            length = float(np.sqrt(12 * (var_y + var_x + span) / 2 + 1))
            width = float(np.sqrt(12 * max(0, (var_y + var_x - span) / 2) + 1))
            outer = expand(box, prob.shape, 3)
            local = np.zeros(prob[outer].shape, bool)
            local[relative(box, outer)] = mask
            ring = ndi.binary_dilation(local, structure=S8, iterations=3) & ~local
            values = prob[box][mask]
            mean = float(means[label])
            contrast = mean - float(prob[outer][ring].mean()) if ring.any() else 0.
            agreement, coverage = None, 0.
            if pb is not None:
                y0, y1, x0, x1 = crop
                valid = (abs_y >= y0) & (abs_y < y1) & (abs_x >= x0) & (abs_x < x1)
                coverage = float(valid.mean())
                if valid.any():
                    vy, vx = abs_y[valid], abs_x[valid]
                    agreement = float(1 - np.abs(p039[vy, vx] - pb[vy-y0, vx-x0]).mean())
            node = Candidate(f"{level}:{label}", level, threshold, box, mask, area, mean, float(peaks[label]),
                             float(np.quantile(values, .9)), float((values >= .75).mean()), length, width,
                             length / width, contrast, agreement, coverage, encode_roi(mask, box, prob.shape), parent)
            index = len(all_nodes)
            if parent is not None:
                all_nodes[parent].children.append(index)
            current[int(label)] = index
            all_nodes.append(node)
        previous_labels, previous_nodes = labels, current

    for node in all_nodes:
        ratios = [all_nodes[i].area / node.area for i in node.children]
        if node.parent is not None:
            ratios.append(node.area / all_nodes[node.parent].area)
        node.stability = max(ratios, default=0.)
        size = float(np.clip(np.log2(node.area / 200) / 3, -1, 1))
        aspect = float(np.clip(np.log2(node.elongation) / 3, 0, 1))
        # Fixed training-free utility. The per-instance cost prevents automatic reward for splitting.
        node.basic = .55 * node.mean + .30 * node.q90 + .15 * node.peak + .08 * size - settings["instance_cost"]
        agreement = 0. if node.agreement is None else (node.agreement - .80) * node.agreement_coverage
        node.quality = (node.basic + .08 * (node.stability - .5) + .08 * (node.contrast - .30)
                        + .04 * (node.high_fraction - .45) + .05 * (aspect - .5) + .08 * agreement)
    return all_nodes


def greedy(nodes, shape):
    occupied = np.zeros(shape, bool)
    selected = []
    for node in sorted(nodes, key=lambda n: (-n.basic, -n.area, n.key)):
        if node.basic <= 0:
            break
        region = occupied[node.box]
        if not np.any(region & node.mask):
            selected.append(node)
            region |= node.mask
    return selected


def tree_select(nodes, score, agreement_weight=1.):
    \"\"\"Maximum utility antichain over sampled level-set trees, with an empty option.\"\"\"
    solutions = [None] * len(nodes)
    for i in range(len(nodes) - 1, -1, -1):
        node = nodes[i]
        total = sum(solutions[j][0] for j in node.children)
        own = getattr(node, score)
        if score == "quality" and node.agreement is not None:
            own -= (1 - agreement_weight) * .08 * (node.agreement - .80) * node.agreement_coverage
        if own > 0 and own >= total:
            solutions[i] = own, [node]
        else:
            solutions[i] = total, [n for j in node.children for n in solutions[j][1]]
    return [n for i, node in enumerate(nodes) if node.parent is None for n in solutions[i][1]]


def close_selected(prob, selected, radius, selective, settings):
    occupied = np.zeros(prob.shape, bool)
    for node in selected:
        occupied[node.box] |= node.mask
    result = []
    for node in sorted(selected, key=lambda n: (-n.quality, -n.area, n.key)):
        box = expand(node.box, prob.shape, radius + 1)
        original = np.zeros(prob[box].shape, bool)
        original[relative(node.box, box)] = node.mask
        region = occupied[box]
        other = region & ~original
        candidate = ndi.binary_closing(original, structure=np.ones((2 * radius + 1,) * 2, bool))
        addition = candidate & ~original & ~other
        if selective:
            addition &= (prob[box] >= settings["repair_min_probability"])
            addition &= ~ndi.binary_dilation(other, structure=S8)
        # Only accept small additions that stay attached to their original instance.
        limit = max(1, int(node.area * settings["repair_max_fraction"]))
        size = int(addition.sum())
        if 0 < size <= limit:
            grown = ndi.binary_propagation(original, structure=S8, mask=original | addition)
            region |= grown
            result.append((node, encode_roi(grown, box, prob.shape), int(grown.sum()) - node.area))
        else:
            result.append((node, node.rle, 0))
    return result


def structural(prob, settings, p039=None, pb=None, crop=None):
    nodes = candidates(prob, settings, p039, pb, crop)
    basic_tree = tree_select(nodes, "basic")
    selected = tree_select(nodes, "quality")
    plain = lambda choices: [(node, node.rle, 0) for node in choices]
    stages = {
        "multithreshold": plain(greedy(nodes, prob.shape)),
        "tree": plain(basic_tree),
        "quality": plain(selected),
        "small_close": close_selected(prob, selected, 1, False, settings),
        "selective_close": close_selected(prob, selected, 2, True, settings),
    }
    if pb is not None:
        stages["quality_no_agreement"] = plain(tree_select(nodes, "quality", agreement_weight=0.))
    return nodes, stages
""",
    'run': """\"\"\"Run a frozen baseline and paired Solar S1.0 ablations.\"\"\"

from collections import defaultdict
import csv
from datetime import datetime, timezone
import gzip
import json
from pathlib import Path
import random
import re
import shutil
import tempfile
import traceback

import numpy as np

from diagnostics import (annotations, apply_calibration, area_stats, calibration_fit, complement, evaluate_photo,
                         mask_counts, pooled, reliability, render_panel, require_disjoint_fast, sample_center,
                         save_candidates)
from probability import (CROP, SHAPE, blend_contract, cache_record, discover_039, ensure_p039, gray_image, load_blend,
                         load_p039, load_s0, metadata_for, save_npz, sha256, write_json)
from reconstruct import KEEP_039, base_cc, feather_weights, fuse, structural


ANN = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
STEM = re.compile(r"[0-9]{14}[A-Z][a-z]")
BASELINE_AUDIT = {"authors_8tta_replay_rows": 1177, "authors_8tta_replay_nonempty_photos": 177,
                  "authors_8tta_replay_sha256_prefix": "3ada6c4418974392",
                  "authors_public_039_submission_sha256_prefix": "eb87c29dfb233d3d"}


def one(paths, label):
    found = sorted(set(paths))
    if len(found) != 1:
        raise FileNotFoundError(f"Expected one {label}, found {found}")
    return found[0]


def official_inputs(root, folds, settings):
    test_dir = one((p for p in root.rglob("test_images") if p.is_dir() and
                    p.parent.parent.name == "MAGFiLO_1.0_Kaggle_2026"), "official MAGFiLO test_images")
    files = sorted(p for p in test_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg"})
    test = {p.stem: p for p in files}
    if len(files) != 180 or len(test) != 180 or any(not STEM.fullmatch(s) for s in test) or set(test) & set(folds):
        raise ValueError("Expected 180 distinct official test stems outside the 707 training photos")
    if not settings["oof_photos"]:
        return test, {}, {}, {}, set(), None
    ann = one(root.rglob(ANN), "official train annotations")
    records, names = annotations(ann, SHAPE)
    if set(folds) != set(records) or any(not STEM.fullmatch(s) for s in names):
        raise ValueError("The Dariush fold roster differs from official training annotations")
    day_folds, dates = defaultdict(set), defaultdict(list)
    for stem, fold in sorted(folds.items()):
        day_folds[stem[:8]].add(fold)
        dates[stem[:8]].append(stem)
    clean_days = {day for day in dates if len(day_folds[day]) == 1}
    # Prefer demonstrably held-out dates. Retain the fold-level exposure flag if extra days are needed.
    rng = random.Random(settings["seed"])
    clean, mixed = sorted(clean_days), sorted(set(dates) - clean_days)
    rng.shuffle(clean)
    rng.shuffle(mixed)
    chosen = [rng.choice(dates[day]) for day in (clean + mixed)[:settings["oof_photos"]]]
    history = {s: ann.parent / "train_images" / names[s] for s in sorted(chosen)}
    if any(not path.is_file() for path in history.values()):
        raise FileNotFoundError("Some OOF photos from the official annotations are not mounted")
    return test, history, records, dates, clean_days, ann


def audit_blend(root, settings, test, history, meta, dates, clean_days):
    path = Path(settings["blend_manifest"]) if settings["blend_manifest"] else None
    if path is None:
        candidates = sorted(root.rglob("blend_v10_manifest.json"))
        if len(candidates) > 1:
            return None, {}, {"status": "ambiguous", "paths": list(map(str, candidates))}
        path = candidates[0] if candidates else None
    if path is None:
        return None, {}, {"status": "missing_exact_v10_probabilities", "test_covered": 0,
                          "reason": "Known V10 output has CSV/validation statistics; no probability maps or checkpoint."}
    try:
        manifest = blend_contract(path)
    except (ValueError, OSError, TypeError, KeyError) as exc:
        return None, {}, {"status": "invalid_manifest", "reason": str(exc), "path": str(path)}
    ready, issues, roles = set(), {}, {}
    for stem in sorted(test.keys() | history.keys()):
        row = manifest["images"].get(stem)
        if row is None:
            issues[stem] = "missing_photo"
            continue
        try:
            load_blend(path.parent, row, meta[stem]["image_sha256"])
            ready.add(stem)
        except (ValueError, OSError, EOFError, KeyError, TypeError) as exc:
            issues[stem] = str(exc)
            continue
        if stem in history:
            day = stem[:8]
            role = row.get("diagnostic_role")
            fit_dates = row.get("blend_fit_dates")
            digest = row.get("checkpoint_sha256", "")
            # A held-out date must be absent from fitting, early stopping and threshold selection.
            if (role in {"calibration", "evaluation"} and day in clean_days and isinstance(fit_dates, list)
                    and fit_dates and all(isinstance(d, str) and re.fullmatch(r"[0-9]{8}", d) and d in dates for d in fit_dates)
                    and row.get("fit_roster_includes_validation_and_selection") is True
                    and day not in fit_dates and isinstance(digest, str) and re.fullmatch(r"[a-f0-9]{64}", digest)):
                roles[stem] = role
    coverage = len(set(test) & ready)
    status = "ready" if coverage == len(test) else "incomplete_exact_v10_test_coverage"
    audit = {"status": status, "manifest": str(path), "manifest_sha256": sha256(path), "test_covered": coverage,
             "history_covered": len(set(history) & ready), "history_roles": roles, "issues": issues,
             "source_url": manifest["source_url"], "provenance_note": manifest["provenance_note"],
             "verification": "Map/image hashes and geometry checked; model identity and exposure depend on producer's attestation."}
    return (path.parent, manifest["images"]) if ready else None, roles, audit


def submit(path, predictions):
    with path.open("w", newline="", encoding="ascii") as stream:
        writer = csv.writer(stream, lineterminator="\\n")
        writer.writerow(("filament_id", "segmentation_rle"))
        for stem, masks in sorted(predictions.items()):
            require_disjoint_fast(masks, SHAPE)
            if not STEM.fullmatch(stem):
                raise ValueError(f"Invalid submission photo ID: {stem}")
            for number, mask in enumerate(masks, 1):
                rle = mask_counts(mask)
                if not rle or any(not 48 <= ord(ch) <= 111 for ch in rle):
                    raise ValueError(f"Invalid COCO counts for {stem}")
                writer.writerow((f"{stem}_{number}", rle))
    return {"sha256": sha256(path), "path": path.name, "rows": sum(map(len, predictions.values())),
            "nonempty_photos": sum(bool(m) for m in predictions.values()), "test_photos": len(predictions)}


def promote_submission(source, output, settings):
    destinations = [output / "submission.csv"]
    if settings["kaggle_submission_path"]:
        destinations.append(Path(settings["kaggle_submission_path"]))
    for destination in destinations:
        destination.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.NamedTemporaryFile(dir=destination.parent, delete=False) as stream, source.open("rb") as data:
            temp = Path(stream.name)
            shutil.copyfileobj(data, stream)
        temp.replace(destination)


def get_pb(blend, stem, meta):
    directory, roster = blend
    return load_blend(directory, roster[stem], meta[stem]["image_sha256"])


def calibrate(blend, roles, records, paths, meta, feather, settings):
    result = {"probability_params": (1., 0.), "base_keep": tuple(settings["blend_keep_candidates"][0]),
              "status": "fixed_fusion_rules_no_verified_calibration_cohort", "photo_count": 0}
    chosen = sorted(s for s, role in roles.items() if role == "calibration")
    if len(chosen) < settings["min_calibration_dates"]:
        return result
    rng = np.random.default_rng(settings["seed"])
    samples, truth = [], []
    for stem in chosen:
        pb = get_pb(blend, stem, meta)
        yy, xx, targets = sample_center(records[stem], SHAPE, CROP, settings["pixel_samples"], rng)
        samples.append(pb[yy-CROP[0], xx-CROP[2]])
        truth.append(targets)
    observed = np.concatenate(truth)
    if observed.sum() < 25 or (1 - observed).sum() < 25:
        result["status"] = "fixed_fusion_rules_insufficient_calibration_pixels"
        return result
    params = calibration_fit(np.concatenate(samples), observed)
    grid = [tuple(keep) for keep in settings["blend_keep_candidates"]]
    rows = {keep: [] for keep in grid}
    for stem in chosen:
        raw, _ = load_p039(paths[stem], meta[stem])
        pb = apply_calibration(get_pb(blend, stem, meta), params)
        fused = fuse(raw, pb, CROP, feather, settings["primary_alpha"])
        for keep in grid:
            masks = base_cc(fused, keep)
            rows[keep].append(evaluate_photo(records[stem], masks, masks))
    scores = {keep: pooled(values)["pq"] for keep, values in rows.items()}
    best = max(grid, key=scores.get)
    return {"status": "fitted_on_separate_declared_dual_oof_dates", "probability_params": params, "base_keep": best,
            "photo_count": len(chosen), "photos": chosen, "search_alpha": settings["primary_alpha"],
            "tune_only_scores": [{"keep": keep, "pq": scores[keep]} for keep in grid]}


def alpha_suffix(alpha):
    return f"a{round(100*alpha):02d}"


def stage_masks(stages, prefix, result, oracles, nodes):
    pool = [node.rle for node in nodes]
    for stage, rows in stages.items():
        name = f"{prefix}_{stage}"
        result[name] = [rle for _, rle, _ in rows]
        oracles[name] = pool


def summarize(rows):
    summary = []
    by_group = defaultdict(list)
    paired_rows = {}
    for row in rows:
        by_group[row["scope"], row["experiment"]].append(row)
        paired_rows[row["scope"], row["experiment"], row["stem"]] = row
    for (scope, name), items in sorted(by_group.items()):
        areas = [area for row in items for area in row["areas"]]
        stat = {"scope": scope, "experiment": name, "photos": len(items), "instances": len(areas),
                "median_area": float(np.median(areas)) if areas else None,
                "area_q10": float(np.quantile(areas, .1)) if areas else None,
                "area_q90": float(np.quantile(areas, .9)) if areas else None}
        if "pq" in items[0]:
            stat.update(pooled(items))
            paired = [paired_rows[scope, "S0", row["stem"]] for row in items]
            stat["paired_s0_pq"] = pooled(paired)["pq"]
            stat["paired_delta_pq"] = stat["pq"] - stat["paired_s0_pq"]
            delta = [row["pq"] - other["pq"] for row, other in zip(items, paired)]
            stat.update(photos_improved=sum(v > 1e-9 for v in delta), photos_regressed=sum(v < -1e-9 for v in delta))
            for row, gain in zip(items, delta):
                row["delta_photo_pq_vs_s0"] = gain
            comparisons = {"S1": ("S0_float_control",), "S2": ("S2_a00",), "S3": ("S1", "S2")}
            for other in comparisons.get(name, ()):
                matched = [paired_rows.get((scope, other, row["stem"])) for row in items]
                if all(row is not None for row in matched):
                    stat[f"paired_delta_pq_vs_{other}"] = stat["pq"] - pooled(matched)["pq"]
        summary.append(stat)
    return summary


def tabulate(path, rows):
    columns = list(dict.fromkeys(k for row in rows for k in row))
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


def run(settings):
    random.seed(settings["seed"])
    np.random.seed(settings["seed"])
    root, working = Path(settings["input_root"]), Path(settings["output_root"])
    output = working / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
    output.mkdir(parents=True)
    write_json(output / "settings.json", settings)
    try:
        execute(root, working, output, settings)
    except Exception:
        write_json(output / "status.json", {"status": "failed", "traceback": traceback.format_exc(),
                                            "baseline_exists": (output / "submission_S0.csv").is_file()})
        raise


def execute(root, working, output, settings):
    receipt_path, receipt, models, folds = discover_039(root, settings["dariush_receipt"])
    receipt_sha, folds_sha = sha256(receipt_path), sha256(receipt_path.parent / "folds.csv")
    oof_issue = None
    try:
        test, history, records, dates, clean_days, ann = official_inputs(root, folds, settings)
    except (ValueError, KeyError, OSError, TypeError) as exc:
        if not settings["oof_photos"]:
            raise
        test, history, records, dates, clean_days, ann = official_inputs(root, folds, settings | {"oof_photos": 0})
        oof_issue = str(exc)
        print(f"Optional fold-OOF inputs unavailable. Continuing with all official test photos: {oof_issue}", flush=True)
    all_photos = test | history
    meta = {s: metadata_for(s, sha256(p), models, folds.get(s), receipt_sha, folds_sha) for s, p in all_photos.items()}
    blend, roles, blend_audit = audit_blend(root, settings, test, history, meta, dates, clean_days)
    audit = {"test_photos": len(test), "oof_photos": len(history), "oof_issue": oof_issue,
             "receipt_path": str(receipt_path),
             "receipt_sha256": receipt_sha, "folds_sha256": folds_sha,
             "annotations_sha256": sha256(ann) if ann else None, "blend": blend_audit,
             "oof_fold_and_date": {s: {"fold": folds[s], "date_held_out_in_author_roster": s[:8] in clean_days}
                                   for s in history},
             "authors_frozen_recipe_evidence": BASELINE_AUDIT, "reference_submission": receipt["runs"]["v6"]["submission_file"]}
    write_json(output / "input_audit.json", audit)
    print(f"Input audit: {len(test)} test, {len(history)} fold-OOF, Blend {blend_audit['status']}", flush=True)
    if settings["audit_only"]:
        absent = [m["file"] for m in models if not (receipt_path.parent / m["file"]).is_file()]
        write_json(output / "status.json", {"status": "audit_complete", "missing_dariush_checkpoints": absent})
        return

    cache = working / "p039_cache" / f"{receipt_sha[:16]}_{folds_sha[:12]}"
    paths, native_paths, inference = ensure_p039(cache, all_photos, meta, receipt_path, models, settings)
    write_json(output / "p039_cache_manifest.json", {"inference": inference, "maps": {
               s: cache_record(native_paths[s], s in paths) for s in all_photos if s in native_paths}})
    missing_test_s0 = sorted(test.keys() - native_paths.keys())
    if missing_test_s0:
        raise FileNotFoundError(f"S0 is missing {len(missing_test_s0)} test maps: {missing_test_s0[:5]}")
    s0 = {s: base_cc(load_s0(native_paths[s], meta[s]), KEEP_039, quantized=True) for s in test}
    baseline = submit(output / "submission_S0.csv", s0)
    promote_submission(output / "submission_S0.csv", output, settings)
    write_json(output / "baseline.json", baseline | {"reference": BASELINE_AUDIT})
    print(f"S0 saved: {baseline['rows']} rows on {baseline['nonempty_photos']} test photos", flush=True)
    missing_test_float = sorted(test.keys() - paths.keys())
    if missing_test_float:
        rows = [{"scope": "unlabelled_test", "stem": s, "experiment": "S0", **area_stats(masks)}
                for s, masks in s0.items()]
        summary = summarize(rows)
        tabulate(output / "summary.csv", summary)
        write_json(output / "submissions.json", {"S0": baseline})
        write_json(output / "summary.json", {"table": summary, "core": {"S0": baseline, "S1": None, "S2": None, "S3": None},
                   "blend_status": blend_audit["status"], "public_lb_measured_here": False,
                   "promoted_submission": "S0", "missing_test_float": missing_test_float})
        write_json(output / "status.json", {"status": "complete_s0_only", "missing_test_float": missing_test_float,
                                            "output": str(output)})
        print(f"S0 is complete. {len(missing_test_float)} test photos lack valid float maps; see p039_cache_manifest.json.",
              flush=True)
        return
    missing_history = sorted(history.keys() - paths.keys())
    if missing_history:
        print(f"Omitting {len(missing_history)} fold-OOF photos without a valid P039; paired results use the same remaining photos.",
              flush=True)
    history = {s: p for s, p in history.items() if s in paths}
    roles = {s: role for s, role in roles.items() if s in history}
    all_photos = test | history
    s0.update({s: base_cc(load_s0(native_paths[s], meta[s]), KEEP_039, quantized=True) for s in history})

    alpha = tuple(float(v) for v in settings["alphas"])
    if sorted(set(alpha)) != list(alpha) or not alpha or alpha[0] != 0 or alpha[-1] > 1:
        raise ValueError("Alpha values must start at zero and increase within [0, 1]")
    primary = settings["primary_alpha"]
    if primary <= 0 or primary not in alpha or len(set(map(alpha_suffix, alpha))) != len(alpha):
        raise ValueError("Primary alpha must be in the sweep, with unique percent suffixes")
    feather = feather_weights(SHAPE, CROP, settings["feather_px"])
    full_fusion = blend_audit["status"] == "ready"
    calibration = calibrate(blend, roles, records, paths, meta, feather, settings)
    write_json(output / "calibration.json", calibration)
    fused_dir, candidate_dir = output / "fused_cache", output / "candidates"
    if full_fusion or roles:
        fused_dir.mkdir()
    candidate_dir.mkdir()
    predictions = defaultdict(dict)
    rows, prob_rows = [], []
    examples = defaultdict(list)
    settings_digest = sha256(output / "settings.json")
    rng = np.random.default_rng(settings["seed"])

    def describe(scope, stem, results, oracles):
        for name, masks in results.items():
            require_disjoint_fast(masks, SHAPE)
            row = {"scope": scope, "stem": stem, "experiment": name, **area_stats(masks)}
            if scope == "unlabelled_test":
                predictions[name][stem] = masks
            else:
                row.update(evaluate_photo(records[stem], masks, oracles.get(name, masks),
                                          settings["small_area"], settings["structural_iou_link"]))
            rows.append(row)

    for i, stem in enumerate(all_photos):
        is_test = stem in test
        scope = ("unlabelled_test" if is_test else
                 "dariush_fold_oof_date_excluded" if stem[:8] in clean_days else "dariush_fold_oof_date_exposed")
        raw, _ = load_p039(paths[stem], meta[stem])
        result = {"S0": s0[stem], "S0_float_control": base_cc(raw, KEEP_039)}
        oracles = {}
        nodes, stages = structural(raw, settings)
        changes = save_candidates(candidate_dir / f"{stem}_S1.json.gz", nodes, stages)
        stage_masks(stages, "S1", result, oracles, nodes)
        result["S1"], oracles["S1"] = result["S1_quality"], oracles["S1_quality"]
        describe(scope, stem, result, oracles)
        slices = np.s_[CROP[0]:CROP[1], CROP[2]:CROP[3]]
        center = raw[slices]
        record = {"stem": stem, "scope": scope, "p039_center": reliability(center),
                  "structural_stage_changes": {stage: {"added": len(change["added_ids"]),
                                                       "removed": len(change["removed_ids"]),
                                                       "repaired": len(change["repaired"])}
                                               for stage, change in changes.items()}}
        sampled = None
        if not is_test:
            sampled = sample_center(records[stem], SHAPE, CROP, settings["pixel_samples"], rng)
            yy, xx, truth = sampled
            record["p039_reliability"] = reliability(raw[yy, xx], truth)
        example = {"stem": stem, "s0": s0[stem], "s1": result["S1"]}
        if not is_test:
            pair = [r for r in rows[-len(result):] if r["experiment"] in {"S0", "S1"}]
            example["gain"] = pair[1]["pq"] - pair[0]["pq"]
            examples[scope].append(example)

        role = roles.get(stem)
        if (is_test and full_fusion) or role in {"evaluation", "calibration"}:
            pb_original = get_pb(blend, stem, meta)
            pb = apply_calibration(pb_original, calibration["probability_params"])
            record.update({"pb_center": reliability(pb_original), "pb_calibrated_center": reliability(pb), "weights": {}})
            if sampled is not None:
                yy, xx, truth = sampled
                sample = (yy - CROP[0], xx - CROP[2])
                record["pb_reliability"] = reliability(pb_original[sample], truth)
                record["pb_calibrated_reliability"] = reliability(pb[sample], truth)
            archive = {"pb": pb_original}
            if calibration["probability_params"] != (1., 0.):
                archive["pb_calibrated"] = pb
            dual_scope = "unlabelled_test" if is_test else f"dual_oof_{role}_producer_attested"
            if not is_test:
                describe(dual_scope, stem, {"S0": s0[stem], "S1": result["S1"]}, {"S1": oracles["S1"]})
            for weight in alpha:
                suffix = alpha_suffix(weight)
                fused = fuse(raw, pb, CROP, feather, weight)
                details = complement(raw, pb, fused, CROP, KEEP_039[0])
                if weight:
                    archive[f"pfused_{suffix}"] = fused
                details["fused_center_histogram"] = reliability(fused[slices])
                if sampled is not None:
                    details["fused_reliability"] = reliability(fused[yy, xx], truth)
                    prior = raw[yy, xx] > KEEP_039[0]
                    revised = fused[yy, xx] > KEEP_039[0]
                    details["sampled_annotator_fg_mass_added"] = float(truth[revised & ~prior].sum())
                    details["sampled_annotator_fg_mass_lost"] = float(truth[prior & ~revised].sum())
                record["weights"][suffix] = details
                names = {f"S2_{suffix}": base_cc(fused, calibration["base_keep"])}
                if weight:
                    fnodes, fstages = structural(fused, settings, raw, pb, CROP)
                    save_candidates(candidate_dir / f"{stem}_S3_{suffix}.json.gz", fnodes, fstages)
                    foracles = {}
                    stage_masks(fstages, f"S3_{suffix}", names, foracles, fnodes)
                    names[f"S3_{suffix}"] = names[f"S3_{suffix}_quality"]
                    foracles[f"S3_{suffix}"] = foracles[f"S3_{suffix}_quality"]
                else:
                    # Zero fusion also zeroes the two-model quality feature.
                    names["S3_a00"], foracles = result["S1"], {"S3_a00": oracles["S1"]}
                if weight == primary:
                    names["S2"], names["S3"] = names[f"S2_{suffix}"], names[f"S3_{suffix}"]
                    foracles["S3"] = foracles[f"S3_{suffix}"]
                    example["s2"], example["s3"] = names["S2"], names["S3"]
                    if is_test:
                        example["gap"] = details["fused_added_pixels"] + details["fused_lost_pixels"]
                        examples["test_disagreement"].append(example)
                    elif role == "evaluation":
                        examples[dual_scope].append(example | {"gain": evaluate_photo(records[stem], names["S3"])["pq"] -
                                                                          evaluate_photo(records[stem], s0[stem])["pq"]})
                describe(dual_scope, stem, names, foracles)
            archive["provenance"] = np.array(json.dumps({"stem": stem, "p039": meta[stem],
                "blend_manifest_sha256": blend_audit["manifest_sha256"], "settings_sha256": settings_digest,
                "crop_yxyx": CROP, "feather_px": settings["feather_px"], "alpha_zero_is_p039": True,
                "alpha": alpha, "calibration_params": calibration["probability_params"]}, sort_keys=True))
            save_npz(fused_dir / f"{stem}.npz", **archive)
        prob_rows.append(record)
        if (i + 1) % 20 == 0:
            print(f"Postprocessing: {i+1}/{len(all_photos)} photos", flush=True)

    submissions = {"S0": baseline}
    for name, masks in sorted(predictions.items()):
        if name != "S0" and set(masks) == set(test):
            submissions[name] = submit(output / f"submission_{name}.csv", masks)
    summary = summarize(rows)
    tabulate(output / "summary.csv", summary)
    with gzip.open(output / "per_photo.jsonl.gz", "wt", encoding="utf-8", compresslevel=3) as stream:
        for row in rows:
            stream.write(json.dumps(row, allow_nan=False) + "\\n")
    with gzip.open(output / "probability_diagnostics.jsonl.gz", "wt", encoding="utf-8", compresslevel=3) as stream:
        for row in prob_rows:
            stream.write(json.dumps(row, allow_nan=False) + "\\n")

    example_dir = output / "examples"
    example_dir.mkdir()
    for scope, values in sorted(examples.items()):
        if scope == "test_disagreement":
            selected = [("largest_disagreement", max(values, key=lambda row: row["gap"]))]
        else:
            best, worst = max(values, key=lambda r: r["gain"]), min(values, key=lambda r: r["gain"])
            selected = [("best_paired", best), ("worst_paired", worst)] if best is not worst else [("paired_example", best)]
        for label, item in selected:
            stem = item["stem"]
            raw, _ = load_p039(paths[stem], meta[stem])
            prob_panels, mask_panels = [("P039", raw)], [("S0", item["s0"]), ("S1", item["s1"])]
            if "s3" in item and (fused_dir / f"{stem}.npz").is_file():
                with np.load(fused_dir / f"{stem}.npz", allow_pickle=False) as pack:
                    pb_canvas = np.zeros(SHAPE, np.float32)
                    pb_canvas[slices] = pack["pb"]
                    prob_panels += [("Pb center", pb_canvas), ("Fused", pack[f"pfused_{alpha_suffix(primary)}"])]
                mask_panels += [("S2", item["s2"]), ("S3", item["s3"])]
            if stem in records:
                mask_panels += [(f"GT {rid.split('-')[0]}", gt) for rid, gt in sorted(records[stem].items())]
            render_panel(example_dir / f"{scope}_{label}_{stem}.png", stem, gray_image(all_photos[stem]),
                         prob_panels, mask_panels)
    write_json(output / "submissions.json", submissions)
    requested = settings["selected_submission"]
    promoted = requested if requested in submissions and submissions[requested]["rows"] else "S0"
    if promoted != "S0":
        promote_submission(output / submissions[promoted]["path"], output, settings)
    elif requested != "S0":
        print(f"Requested {requested} has no complete nonempty submission; keeping S0", flush=True)
    write_json(output / "summary.json", {"table": summary, "core": {name: submissions.get(name) for name in
        ("S0", "S1", "S2", "S3")}, "blend_status": blend_audit["status"], "public_lb_measured_here": False,
        "primary_alpha": primary, "requested_submission": requested, "promoted_submission": promoted,
        "missing_oof_float": missing_history,
        "paired_selection": "S1/S3 use fixed quality selection; closes are separate arms."})
    write_json(output / "status.json", {"status": "complete" if full_fusion else "complete_p039_only",
                                       "submission_files": len(submissions), "output": str(output)})
    for row in summary:
        if row["experiment"] in {"S0", "S1", "S2", "S3"}:
            print(row, flush=True)
    print(f"Solar S1.0 outputs: {output}", flush=True)
""",
}

if importlib.util.find_spec("pycocotools") is None:
    if not CONFIG["install_missing_pycocotools"]:
        raise ImportError("Kaggle runtime requires pycocotools")
    subprocess.run([sys.executable, "-m", "pip", "install", "pycocotools==2.0.11"], check=True)
base = Path(CONFIG["scratch_root"])
base.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(prefix="source_", dir=base) as folder:
    for name, content in MODULES.items():
        (Path(folder) / f"{name}.py").write_text(content, encoding="utf-8")
    entry = 'import json, sys; from run import run; settings = json.loads(sys.argv[1]); settings["code_sha256"] = sys.argv[2]; run(settings)'
    subprocess.run([sys.executable, "-c", entry, json.dumps(CONFIG), CODE_SHA256], cwd=folder,
                   env=dict(os.environ, PYTHONUNBUFFERED="1"), check=True)
